In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
!curl -I https://github.com/uoi1398/bird-drone-dualstream

HTTP/2 200 
date: Sat, 16 May 2026 06:26:50 GMT
content-type: text/html; charset=utf-8
vary: X-PJAX, X-PJAX-Container, Turbo-Visit, Turbo-Frame, X-Requested-With, Sec-Fetch-Site,Accept-Encoding, Accept, X-Requested-With
etag: W/"bbd394264410b1ecf070f353a195de79"
cache-control: max-age=0, private, must-revalidate
strict-transport-security: max-age=31536000; includeSubdomains; preload
x-frame-options: deny
x-content-type-options: nosniff
x-xss-protection: 0
referrer-policy: no-referrer-when-downgrade
content-security-policy: default-src 'none'; base-uri 'self'; child-src github.githubassets.com github.com/assets-cdn/worker/ github.com/assets/ gist.github.com/assets-cdn/worker/; connect-src 'self' uploads.github.com www.githubstatus.com collector.github.com raw.githubusercontent.com api.github.com github-cloud.s3.amazonaws.com github-production-repository-file-5c1aeb.s3.amazonaws.com github-production-upload-manifest-file-7fdce7.s3.amazonaws.com github-production-user-asset-6210df.s3.amaz

In [5]:
%cd /content

import os

REPO_DIR = "/content/bird-drone-dualstream"
REPO_URL = "https://github.com/uoi1398/bird-drone-dualstream.git"

if os.path.exists(REPO_DIR):
    print("Removing old repo...")
    !rm -rf /content/bird-drone-dualstream

print("Testing repo access...")
!git ls-remote {REPO_URL}

print("Cloning repo...")
!git clone {REPO_URL}

%cd /content/bird-drone-dualstream

print("Current directory:")
!pwd

print("Files:")
!ls -la

/content
Removing old repo...
Testing repo access...
4f978fcec395b1912c62cb516e24dc26a0b105fe	HEAD
4f978fcec395b1912c62cb516e24dc26a0b105fe	refs/heads/main
Cloning repo...
Cloning into 'bird-drone-dualstream'...
remote: Enumerating objects: 40, done.
remote: Counting objects: 100% (40/40), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 40 (delta 12), reused 27 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (40/40), 21.47 KiB | 628.00 KiB/s, done.
Resolving deltas: 100% (12/12), done.
/content/bird-drone-dualstream
Current directory:
/content/bird-drone-dualstream
Files:
total 40
drwxr-xr-x 6 root root 4096 May 16 06:28 .
drwxr-xr-x 1 root root 4096 May 16 06:28 ..
drwxr-xr-x 2 root root 4096 May 16 06:28 configs
drwxr-xr-x 8 root root 4096 May 16 06:28 .git
-rw-r--r-- 1 root root 4733 May 16 06:28 .gitignore
drwxr-xr-x 2 root root 4096 May 16 06:28 notebooks
-rw-r--r-- 1 root root 1682 May 16 06:28 README.md
-rw-r--r-- 1 root root   96 May 16 06:28 requ

In [6]:
!pip install -r requirements.txt

In [9]:
%cd /content

# 下载数据集仓库
!git clone https://github.com/DroneDetectionThesis/Drone-detection-dataset.git

# 回到你的项目
%cd /content/bird-drone-dualstream

# 建 data 文件夹
!mkdir -p data

# 把下载的数据集链接到项目 data/raw
!ln -s /content/Drone-detection-dataset data/raw

# 检查
!ls data/raw

/content
fatal: destination path 'Drone-detection-dataset' already exists and is not an empty directory.
/content/bird-drone-dualstream
Data		       Demo_V_DRONE_017.mp4	LICENSE
Demo_IR_DRONE_146.mp4  Drone-detection-dataset	README.md


In [16]:
%cd /content/bird-drone-dualstream
!git pull

/content/bird-drone-dualstream
Already up to date.


In [17]:
!head -5 src/dataset.py
!head -5 src/model.py
!head -5 src/train.py


import cv2
import torch
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights


import os
import json
import yaml
import argparse


In [18]:
import sys
sys.path.append("src")

from dataset import BirdDroneVideoDataset
from model import FrameAverageResNet

print("Imports OK.")

Imports OK.


In [19]:
%cd /content/bird-drone-dualstream

!find /content/Drone-detection-dataset -maxdepth 3 -type d | head -80
!find /content/Drone-detection-dataset/Data -maxdepth 3 -type f | head -80

/content/bird-drone-dualstream
/content/Drone-detection-dataset
/content/Drone-detection-dataset/.git
/content/Drone-detection-dataset/.git/branches
/content/Drone-detection-dataset/.git/refs
/content/Drone-detection-dataset/.git/refs/remotes
/content/Drone-detection-dataset/.git/refs/heads
/content/Drone-detection-dataset/.git/refs/tags
/content/Drone-detection-dataset/.git/hooks
/content/Drone-detection-dataset/.git/info
/content/Drone-detection-dataset/.git/logs
/content/Drone-detection-dataset/.git/logs/refs
/content/Drone-detection-dataset/.git/objects
/content/Drone-detection-dataset/.git/objects/pack
/content/Drone-detection-dataset/.git/objects/info
/content/Drone-detection-dataset/Data
/content/Drone-detection-dataset/Data/Video_V
/content/Drone-detection-dataset/Data/Video_IR
/content/Drone-detection-dataset/Data/Audio
/content/Drone-detection-dataset/Data/Create_a_dataset_from_videos_and_labels.m
/content/Drone-detection-dataset/Data/Video_V/V_DRONE_070_LABELS.mat
/content/D

In [20]:
from pathlib import Path

DATA_ROOT = Path("/content/Drone-detection-dataset/Data")

VIDEO_EXTS = [".mp4", ".avi", ".mov", ".mkv"]

all_videos = []
for ext in VIDEO_EXTS:
    all_videos.extend(DATA_ROOT.rglob(f"*{ext}"))

all_videos = sorted(all_videos)

print("All videos:", len(all_videos))

for p in all_videos[:20]:
    print(p)

All videos: 650
/content/Drone-detection-dataset/Data/Video_IR/IR_AIRPLANE_001.mp4
/content/Drone-detection-dataset/Data/Video_IR/IR_AIRPLANE_002.mp4
/content/Drone-detection-dataset/Data/Video_IR/IR_AIRPLANE_003.mp4
/content/Drone-detection-dataset/Data/Video_IR/IR_AIRPLANE_004.mp4
/content/Drone-detection-dataset/Data/Video_IR/IR_AIRPLANE_005.mp4
/content/Drone-detection-dataset/Data/Video_IR/IR_AIRPLANE_006.mp4
/content/Drone-detection-dataset/Data/Video_IR/IR_AIRPLANE_007.mp4
/content/Drone-detection-dataset/Data/Video_IR/IR_AIRPLANE_008.mp4
/content/Drone-detection-dataset/Data/Video_IR/IR_AIRPLANE_009.mp4
/content/Drone-detection-dataset/Data/Video_IR/IR_AIRPLANE_010.mp4
/content/Drone-detection-dataset/Data/Video_IR/IR_AIRPLANE_011.mp4
/content/Drone-detection-dataset/Data/Video_IR/IR_AIRPLANE_012.mp4
/content/Drone-detection-dataset/Data/Video_IR/IR_AIRPLANE_013.mp4
/content/Drone-detection-dataset/Data/Video_IR/IR_AIRPLANE_014.mp4
/content/Drone-detection-dataset/Data/Video_IR

In [21]:
visible_videos = [
    p for p in all_videos
    if (
        "VIDEO_V" in str(p).upper()
        or "VISIBLE" in str(p).upper()
        or p.name.upper().startswith("V_")
        or "_V_" in p.name.upper()
    )
]

bird_videos = [p for p in visible_videos if "BIRD" in p.name.upper()]
drone_videos = [p for p in visible_videos if "DRONE" in p.name.upper()]

print("Visible videos:", len(visible_videos))
print("Bird visible videos:", len(bird_videos))
print("Drone visible videos:", len(drone_videos))

print("\nExample bird:")
for p in bird_videos[:5]:
    print(p)

print("\nExample drone:")
for p in drone_videos[:5]:
    print(p)

Visible videos: 285
Bird visible videos: 51
Drone visible videos: 114

Example bird:
/content/Drone-detection-dataset/Data/Video_V/V_BIRD_001.mp4
/content/Drone-detection-dataset/Data/Video_V/V_BIRD_002.mp4
/content/Drone-detection-dataset/Data/Video_V/V_BIRD_003.mp4
/content/Drone-detection-dataset/Data/Video_V/V_BIRD_004.mp4
/content/Drone-detection-dataset/Data/Video_V/V_BIRD_005.mp4

Example drone:
/content/Drone-detection-dataset/Data/Video_V/V_DRONE_001.mp4
/content/Drone-detection-dataset/Data/Video_V/V_DRONE_002.mp4
/content/Drone-detection-dataset/Data/Video_V/V_DRONE_003.mp4
/content/Drone-detection-dataset/Data/Video_V/V_DRONE_004.mp4
/content/Drone-detection-dataset/Data/Video_V/V_DRONE_005.mp4


In [23]:
import random
import pandas as pd
import os

SEED = 42
random.seed(SEED)

assert len(bird_videos) == 51, f"Expected 51 bird videos, got {len(bird_videos)}"
assert len(drone_videos) == 114, f"Expected 114 drone videos, got {len(drone_videos)}"

random.shuffle(bird_videos)
random.shuffle(drone_videos)

records = []

def add_records(video_paths, label, label_name, train_n, val_n, test_n):
    assert len(video_paths) == train_n + val_n + test_n

    split_names = (
        ["train"] * train_n +
        ["val"] * val_n +
        ["test"] * test_n
    )

    for path, split in zip(video_paths, split_names):
        records.append({
            "video_id": path.stem,
            "path": str(path),
            "label": label,
            "label_name": label_name,
            "split": split
        })

add_records(
    bird_videos,
    label=0,
    label_name="bird",
    train_n=36,
    val_n=7,
    test_n=8
)

add_records(
    drone_videos,
    label=1,
    label_name="drone",
    train_n=79,
    val_n=18,
    test_n=17
)

manifest = pd.DataFrame(records)

os.makedirs("data/splits", exist_ok=True)

manifest_path = "data/splits/manifest.csv"
manifest.to_csv(manifest_path, index=False)

print("Saved manifest to:", manifest_path)
display(manifest.head())

print(pd.crosstab(manifest["split"], manifest["label_name"]))

Saved manifest to: data/splits/manifest.csv


,video_id,path,label,label_name,split
0,V_BIRD_026,/content/Drone-detection-dataset/Data/Video_V/...,0,bird,train
1,V_BIRD_024,/content/Drone-detection-dataset/Data/Video_V/...,0,bird,train
2,V_BIRD_020,/content/Drone-detection-dataset/Data/Video_V/...,0,bird,train
3,V_BIRD_012,/content/Drone-detection-dataset/Data/Video_V/...,0,bird,train
4,V_BIRD_005,/content/Drone-detection-dataset/Data/Video_V/...,0,bird,train


label_name  bird  drone
split                  
test           8     17
train         36     79
val            7     18


In [24]:
import os
import pandas as pd

df = pd.read_csv("data/splits/manifest.csv")
df["path_exists"] = df["path"].apply(os.path.exists)

print(df["path_exists"].value_counts())
display(df.head())

assert df["path_exists"].all(), "Some video paths do not exist."

path_exists
True    165
Name: count, dtype: int64


,video_id,path,label,label_name,split,path_exists
0,V_BIRD_026,/content/Drone-detection-dataset/Data/Video_V/...,0,bird,train,True
1,V_BIRD_024,/content/Drone-detection-dataset/Data/Video_V/...,0,bird,train,True
2,V_BIRD_020,/content/Drone-detection-dataset/Data/Video_V/...,0,bird,train,True
3,V_BIRD_012,/content/Drone-detection-dataset/Data/Video_V/...,0,bird,train,True
4,V_BIRD_005,/content/Drone-detection-dataset/Data/Video_V/...,0,bird,train,True


In [25]:
import sys
sys.path.append("src")

from dataset import BirdDroneVideoDataset

rgb_ds = BirdDroneVideoDataset(
    csv_path="data/splits/manifest.csv",
    split="train",
    mode="rgb",
    num_frames=16,
    image_size=224,
    degrade_level="none"
)

x, y, video_id = rgb_ds[0]

print(x.shape)
print(y)
print(video_id)

torch.Size([16, 3, 224, 224])
tensor(0)
V_BIRD_026


In [26]:
flow_ds = BirdDroneVideoDataset(
    csv_path="data/splits/manifest.csv",
    split="train",
    mode="flow",
    num_frames=8,
    image_size=160,
    degrade_level="none"
)

x_flow, y_flow, video_id_flow = flow_ds[0]

print(x_flow.shape)
print(y_flow)
print(video_id_flow)

torch.Size([7, 3, 160, 160])
tensor(0)
V_BIRD_026


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import os

os.makedirs("data", exist_ok=True)

# 如果 data/raw 已经存在，可以先删掉软链接
# !rm -rf data/raw

!ln -s /content/drive/MyDrive/datasets/Drone-detection-dataset data/raw

!ls data
!ls data/raw | head